In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import random

np.random.seed(42)
random.seed(42)

def generate_player_data(number_of_samples):
    data = []
    labels = []

    def apply_bounds(stats):
        try:
            stats['kills'] = max(0, int(stats['kills']))
            stats['deaths'] = max(1, int(stats['deaths']))
            stats['accuracy'] = np.clip(stats['accuracy'], 1.0, 100.0)
            stats['headshot_pct'] = np.clip(stats['headshot_pct'], 0.0, 100.0)
            stats['shots_fired'] = max(50, int(stats['shots_fired']))
            stats['shots_hit'] = int(stats['shots_fired'] * (stats['accuracy'] / 100.0))
            stats['survival_rate'] = np.clip(stats['survival_rate'], 0.0, 100.0)
            stats['reaction_time'] = max(10, stats['reaction_time'])
            stats['dmg_per_match'] = max(0, stats['dmg_per_match'])
            stats['avg_kill_dist'] = max(0, stats['avg_kill_dist'])
        except Exception as error:
            print("Error has occurred while applying bounds:", error)

        return stats

    def generate_normal_player():
        stats = {
            'kills': np.random.normal(12, 4),
            'deaths': np.random.normal(15, 3),
            'accuracy': np.random.normal(25, 5),
            'headshot_pct': np.random.normal(20, 7),
            'dmg_per_match': np.random.normal(1500, 400),
            'survival_rate': np.random.normal(35, 10),
            'avg_kill_dist': np.random.normal(25, 8),
            'shots_fired': np.random.normal(400, 100),
            'reaction_time': np.random.normal(250, 30)
        }

        if random.random() < 0.05:
            try:
                stats['kills'] += 15
                stats['headshot_pct'] += 15
            except Exception as error:
                print("Error has occurred while generating normal player:", error)

        return stats

    def generate_experienced_player():
        stats = {
            'kills': np.random.normal(25, 6),
            'deaths': np.random.normal(10, 3),
            'accuracy': np.random.normal(40, 6),
            'headshot_pct': np.random.normal(45, 8),
            'dmg_per_match': np.random.normal(3000, 600),
            'survival_rate': np.random.normal(55, 12),
            'avg_kill_dist': np.random.normal(35, 10),
            'shots_fired': np.random.normal(550, 120),
            'reaction_time': np.random.normal(190, 20)
        }

        try:
            if random.random() < 0.05:
                stats['avg_kill_dist'] = np.random.normal(70, 15)
                stats['accuracy'] += 10
            elif random.random() < 0.05: stats['deaths'] += 10
        except Exception as error:
            print("Error has occurred while generating experienced player:", error)

        return stats

    def generate_cheater_player():
        cheat_style = np.random.choice(['blatant', 'closet', 'wallhack'])

        if cheat_style == 'blatant':
            return {
                'kills': np.random.normal(45, 5),
                'deaths': np.random.normal(3, 2),
                'accuracy': np.random.normal(75, 10),
                'headshot_pct': np.random.normal(85, 5),
                'dmg_per_match': np.random.normal(5000, 500),
                'survival_rate': np.random.normal(80, 10),
                'avg_kill_dist': np.random.normal(40, 15),
                'shots_fired': np.random.normal(600, 100),
                'reaction_time': np.random.normal(80, 15) # Anomaly
            }
        elif cheat_style == 'closet':
            return {
                'kills': np.random.normal(15, 4),
                'deaths': np.random.normal(14, 3),
                'accuracy': np.random.normal(28, 5),
                'headshot_pct': np.random.normal(75, 8), # Anomaly
                'dmg_per_match': np.random.normal(1600, 300),
                'survival_rate': np.random.normal(40, 10),
                'avg_kill_dist': np.random.normal(25, 8),
                'shots_fired': np.random.normal(400, 100),
                'reaction_time': np.random.normal(110, 10) # Anomaly
            }
        else:
            return {
                'kills': np.random.normal(30, 5),
                'deaths': np.random.normal(5, 2),
                'accuracy': np.random.normal(20, 5), # Anomaly
                'headshot_pct': np.random.normal(25, 5),
                'dmg_per_match': np.random.normal(3500, 400),
                'survival_rate': np.random.normal(70, 10),
                'avg_kill_dist': np.random.normal(55, 12),
                'shots_fired': np.random.normal(1000, 200), # Anomaly
                'reaction_time': np.random.normal(230, 30)
            }

    for index in range(number_of_samples):
        try:
            player_type = np.random.choice([0, 1, 2])

            if player_type == 0: raw_stats = generate_normal_player()
            elif player_type == 1: raw_stats = generate_experienced_player()
            else: raw_stats = generate_cheater_player()

            clean_stats = apply_bounds(raw_stats)

            data.append(
                [
                    clean_stats['kills'],
                    clean_stats['deaths'],
                    clean_stats['accuracy'],
                    clean_stats['headshot_pct'],
                    clean_stats['dmg_per_match'],
                    clean_stats['survival_rate'],
                    clean_stats['avg_kill_dist'],
                    clean_stats['shots_fired'],
                    clean_stats['shots_hit'],
                    clean_stats['reaction_time']
                ]
            )
            labels.append(player_type)
        except Exception as error:
            print(f"An error has occurred on lap #{index}:", error)

    columns = ['Kills', 'Deaths', 'Accuracy', 'HeadshotPercent', 'DamagePerMatch', 'SurvivalRate', 'AverageKillDistance', 'ShotsFired', 'ShotsHit', 'ReactionTime']

    return pd.DataFrame(data, columns=columns), np.array(labels)

data_frame, labels = generate_player_data(1000)